# DSMC Shape Optimisation Demo

Optimise a Fourier-parameterised 2-D domain boundary to **minimise the mean squared
distance from the origin** of particles that land inside the inner square after $M$ steps.

**Setup**
- Outer bounding box: $[-1.2,\,1.2]^2$
- Inner observation square: $[-0.35,\,0.35]^2$
- Boundary: $r(\theta;C)=c_0+\sum_{k=1}^N(a_k\cos 2\pi k\theta+b_k\sin 2\pi k\theta)$
- Gradient: **analytic adjoint DSMC** — no finite differences

Run all cells top-to-bottom.


In [ ]:
import sys, os

# Support both JupyterHub layout (adjoint/ next to notebook) and
# local repo layout (src/adjoint/).
_nb_dir  = os.path.dirname(os.path.abspath('__file__'))
_src_dir = os.path.join(os.path.dirname(_nb_dir), 'src')
for _p in [_nb_dir, _src_dir, os.getcwd(), os.path.join(os.getcwd(), '..', 'src')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

from adjoint import ForwardSimulation
from adjoint.boundary_geometry import gamma, radius_r
from adjoint.shape_gradient import (
    shape_gradient, perimeter, perimeter_gradient, dr_dC
)
from adjoint.visualization import (
    animate_simulation, animate_comparison,
    plot_convergence, plot_objective_landscape_1d, plot_final_density
)

print('Imports OK')

In [ ]:
# ── Simulation ────────────────────────────────────────────────────────
N_PARTICLES = 150
N_FOURIER   = 3          # C has 2N+1 = 7 coefficients
DT          = 0.10
N_STEPS     = 20
N_COLL      = N_PARTICLES // 4
SEED_INIT   = 42

# ── Domain ────────────────────────────────────────────────────────────
R_INIT      = 0.80
BOX_HALF    = 1.20
INNER_HALF  = 0.35
R_MAX       = BOX_HALF

# ── Optimisation ──────────────────────────────────────────────────────
N_ITER      = 120
LR          = 2e-3
LAM_PERIM   = 30.0       # strong perimeter enforcement
LAM_BOX     = 8.0
P_TARGET    = 2 * np.pi * R_INIT   # ≈ 5.03

# ── Stability controls ────────────────────────────────────────────────
N_AVG       = 5
G_MAX       = 0.5
C0_MIN      = 0.35
C0_MAX      = 1.10
A_MAX_FRAC  = 0.45

print(f'C length: {2*N_FOURIER+1},  P_target: {P_TARGET:.4f}')
print(f'N_PARTICLES={N_PARTICLES}, N_STEPS={N_STEPS}, N_AVG={N_AVG}, LR={LR}')

In [ ]:
def boundary_pts(C, n=300):
    thetas = np.linspace(0, 1, n)
    return np.array([gamma(t, C) for t in thetas])

def plot_domain(C, ax=None, particles=None, title='', color='royalblue'):
    if ax is None: _, ax = plt.subplots(figsize=(5,5))
    ax.add_patch(plt.Rectangle((-BOX_HALF,-BOX_HALF),2*BOX_HALF,2*BOX_HALF,
        fill=False,edgecolor='black',lw=2,label='Outer box'))
    ax.add_patch(plt.Rectangle((-INNER_HALF,-INNER_HALF),2*INNER_HALF,2*INNER_HALF,
        facecolor='lightyellow',edgecolor='darkorange',lw=1.5,alpha=0.7,label='Inner sq'))
    pts = boundary_pts(C)
    ax.plot(pts[:,0],pts[:,1],color=color,lw=2,label='Boundary')
    if particles is not None:
        ax.scatter(particles[:,0],particles[:,1],s=8,c='crimson',alpha=0.6,zorder=3)
    ax.set_xlim(-BOX_HALF*1.1,BOX_HALF*1.1); ax.set_ylim(-BOX_HALF*1.1,BOX_HALF*1.1)
    ax.set_aspect('equal'); ax.set_title(title,fontsize=11)
    ax.legend(loc='upper right',fontsize=8)
    return ax

print('Helpers ready')


In [ ]:
def compute_L(history):
    """
    Objective: mean squared distance from origin over ALL particles.

    Using all particles (not just those in the inner square) gives a much
    lower-variance gradient estimate, which is essential for the stochastic
    adjoint method to converge.  The inner square is still visualised but
    is not used to filter the loss.
    """
    xM = history.final_positions            # (N, 2)
    return float(np.mean(np.sum(xM**2, axis=1)))


def term_beta(vM, xM):
    return np.zeros_like(vM)


def term_alpha(vM, xM):
    """α_M = ∂L/∂x_M = 2 x_M / N  (for all particles)."""
    return 2.0 * xM / len(xM)


def compute_L_inner(history):
    """Secondary diagnostic: mean |x|² for particles inside the inner square."""
    xM   = history.final_positions
    mask = (np.abs(xM[:, 0]) <= INNER_HALF) & (np.abs(xM[:, 1]) <= INNER_HALF)
    if not mask.any():
        return float('nan')
    return float(np.sum(xM[mask]**2) / mask.sum())


def box_penalty(C, n=200):
    ts = np.linspace(0, 1, n, endpoint=False)
    r  = np.array([radius_r(t, C) for t in ts])
    return float(np.sum(np.maximum(r - R_MAX, 0)**2) / n)


def box_penalty_grad(C, n=200):
    ts = np.linspace(0, 1, n, endpoint=False)
    g  = np.zeros_like(C)
    for t in ts:
        v = max(radius_r(t, C) - R_MAX, 0.0)
        if v > 0:
            g += 2 * v * dr_dC(t, C)
    return g / n


print('Objective functions ready  (L = mean |x_M|² over all particles)')

In [ ]:
rng0 = np.random.default_rng(SEED_INIT)
r0   = rng0.uniform(0.25, 0.70, N_PARTICLES)
th0  = rng0.uniform(0, 2*np.pi, N_PARTICLES)
x0   = np.column_stack([r0 * np.cos(th0), r0 * np.sin(th0)])
v0   = rng0.standard_normal((N_PARTICLES, 2))

# Large initial asymmetry gives a clear starting point far from the optimum.
# a_1 = 0.25 → strongly elliptic (r varies between 0.55 and 1.05).
# The optimizer should drive a_1 toward 0 (near-circular minimum).
C_init    = np.zeros(2 * N_FOURIER + 1)
C_init[0] = R_INIT
C_init[1] = 0.25   # large a_1 perturbation

sim0  = ForwardSimulation(C_init, DT, n_coll_pairs=N_COLL, seed=0)
hist0 = sim0.run(x0.copy(), v0.copy(), N_STEPS)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_domain(C_init, ax=axes[0], particles=x0, title='Initial positions')
plot_domain(C_init, ax=axes[1], particles=hist0.final_positions,
            title=f'After {N_STEPS} steps  |  L = {compute_L(hist0):.4f}')
plt.tight_layout(); plt.show()
print(f'L = {compute_L(hist0):.4f}  (mean |x_M|² over all particles)')

## Gradient Validation
Single particle, zero collisions → fully deterministic → adjoint gradient must
match finite differences to near machine precision.


In [ ]:
C_val = np.array([0.8, 0.12, -0.06, 0.03, 0.09, -0.03, 0.02])
x_val = np.array([[0.15, 0.08]])
v_val = np.array([[0.6,  1.1]])

def L_val(C_):
    s = ForwardSimulation(C_, DT, n_coll_pairs=0, seed=0)
    h = s.run(x_val.copy(), v_val.copy(), N_STEPS)
    return float(np.sum(h.final_positions**2))

sv = ForwardSimulation(C_val, DT, n_coll_pairs=0, seed=0)
hv = sv.run(x_val.copy(), v_val.copy(), N_STEPS)
bv, av = hv.backward_pass(lambda v,x: np.zeros_like(v), lambda v,x: 2*x)
g_adj = shape_gradient(hv, bv, av)

eps  = 1e-6
L0   = L_val(C_val)
g_fd = np.zeros_like(C_val)
for j in range(len(C_val)):
    Cp = C_val.copy(); Cp[j] += eps
    g_fd[j] = (L_val(Cp) - L0) / eps

rel = np.linalg.norm(g_adj - g_fd) / (np.linalg.norm(g_fd) + 1e-10)
print(f'Adjoint:  {g_adj.round(5)}')
print(f'FD:       {g_fd.round(5)}')
print(f'Rel error: {rel:.2e}   (< 1e-4 is good)')


## Optimisation Loop
Gradient descent on $L + \lambda_P(P-P_{\mathrm{tgt}})^2/P_{\mathrm{tgt}}^2 + \lambda_{\mathrm{box}}\cdot\mathrm{BoxPenalty}$.
The gradient norm is tracked to confirm convergence.


In [ ]:
def _project_C(C, c0_min, c0_max, a_max_frac):
    """
    Project C onto the feasible set:
      - c0 ∈ [c0_min, c0_max]
      - each Fourier mode amplitude sqrt(ak² + bk²) ≤ a_max_frac * c0
    This keeps r(θ) > 0 everywhere (star-shapedness) and the boundary inside the box.
    """
    C = C.copy()
    C[0] = np.clip(C[0], c0_min, c0_max)
    N_f = (len(C) - 1) // 2
    A_MAX = a_max_frac * C[0]
    for ki in range(1, N_f + 1):
        ak, bk = C[ki], C[N_f + ki]
        amp = np.sqrt(ak**2 + bk**2)
        if amp > A_MAX:
            scale = A_MAX / amp
            C[ki]       *= scale
            C[N_f + ki] *= scale
    return C


C = C_init.copy()
obj_hist, perim_hist, grad_norm_hist = [], [], []
C_snapshots = [C.copy()]

print(f"{'Iter':>5} | {'L':>8} | {'Perim':>7} | {'|gL|':>8} | {'|g_tot|':>8}")
print('-'*50)

for it in range(N_ITER):
    # ── Average gradient over N_AVG independent DSMC runs ──────────
    L_vals  = []
    gL_vals = []
    for s in range(N_AVG):
        sim_  = ForwardSimulation(C, DT, n_coll_pairs=N_COLL, seed=it * N_AVG + s)
        hist_ = sim_.run(x0.copy(), v0.copy(), N_STEPS)
        L_vals.append(compute_L(hist_))
        b_, a_ = hist_.backward_pass(term_beta, term_alpha)
        gL_vals.append(shape_gradient(hist_, b_, a_))

    L   = float(np.mean(L_vals))
    g_L = np.mean(gL_vals, axis=0)

    obj_hist.append(L)
    perim_hist.append(perimeter(C))
    grad_norm_hist.append(float(np.linalg.norm(g_L)))

    # ── Total gradient: objective + perimeter + box constraints ─────
    P_now  = perim_hist[-1]
    g_P    = perimeter_gradient(C)
    g_box  = box_penalty_grad(C)
    total  = g_L + 2 * LAM_PERIM * (P_now - P_TARGET) / P_TARGET**2 * g_P + LAM_BOX * g_box

    # ── Gradient clipping ───────────────────────────────────────────
    g_norm = float(np.linalg.norm(total))
    if g_norm > G_MAX:
        total = total * (G_MAX / g_norm)

    # ── Gradient step + feasibility projection ──────────────────────
    C = _project_C(C - LR * total, C0_MIN, C0_MAX, A_MAX_FRAC)

    if it % 5 == 0:
        C_snapshots.append(C.copy())
    if it % 10 == 0 or it == N_ITER - 1:
        print(f'{it:5d} | {L:8.4f} | {P_now:7.3f} | {grad_norm_hist[-1]:8.4f} | {g_norm:8.4f}')

C_snapshots.append(C.copy())
C_opt = C.copy()
print(f'\nDone.  L: {obj_hist[0]:.4f} → {obj_hist[-1]:.4f}')
print(f'C_opt: {C_opt.round(4)}')
print(f'Perimeter: {perimeter(C_init):.4f} → {perimeter(C_opt):.4f}  (target {P_TARGET:.4f})')

## Results — Convergence & Boundary Evolution


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Convergence with gradient norm
plot_convergence(obj_hist, grad_norm_hist, ax=axes[0])

# Boundary evolution (colour = iteration)
ax = axes[1]
ax.add_patch(plt.Rectangle((-BOX_HALF,-BOX_HALF),2*BOX_HALF,2*BOX_HALF,
    fill=False,edgecolor='k',lw=2))
ax.add_patch(plt.Rectangle((-INNER_HALF,-INNER_HALF),2*INNER_HALF,2*INNER_HALF,
    facecolor='lightyellow',edgecolor='darkorange',lw=1.5,alpha=0.6))
cmap = plt.cm.viridis; n_s = len(C_snapshots)
for si, Cs in enumerate(C_snapshots):
    c = cmap(si / max(n_s-1, 1))
    pts = boundary_pts(Cs)
    ax.plot(pts[:,0],pts[:,1],color=c,lw=1.2,alpha=0.4+0.6*si/max(n_s-1,1))
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0,N_ITER))
plt.colorbar(sm, ax=ax, label='Iteration')
ax.set_xlim(-BOX_HALF*1.1,BOX_HALF*1.1); ax.set_ylim(-BOX_HALF*1.1,BOX_HALF*1.1)
ax.set_aspect('equal'); ax.set_title('Boundary evolution')

# Initial vs final with final particles
sim_f  = ForwardSimulation(C_opt, DT, n_coll_pairs=N_COLL, seed=0)
hist_f = sim_f.run(x0.copy(), v0.copy(), N_STEPS)
plot_domain(C_init, ax=axes[2], color='grey')
axes[2].plot(boundary_pts(C_opt)[:,0], boundary_pts(C_opt)[:,1],
             color='royalblue', lw=2.5, label='Optimised')
axes[2].scatter(hist_f.final_positions[:,0], hist_f.final_positions[:,1],
                s=10, c='crimson', alpha=0.6, zorder=3)
axes[2].set_title(
    f'Grey=initial, Blue=optimised\nL: {obj_hist[0]:.4f} → {obj_hist[-1]:.4f}')
axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()


## Animation — Particle Simulation
Particles are coloured **red** when inside the inner square, **blue** otherwise.


In [ ]:
# Re-run both simulations with the same seed for fair comparison
sim_init = ForwardSimulation(C_init, DT, n_coll_pairs=N_COLL, seed=99)
hist_init_anim = sim_init.run(x0.copy(), v0.copy(), N_STEPS)

sim_opt  = ForwardSimulation(C_opt,  DT, n_coll_pairs=N_COLL, seed=99)
hist_opt_anim = sim_opt.run(x0.copy(), v0.copy(), N_STEPS)

L_init = compute_L(hist_init_anim)
L_opt  = compute_L(hist_opt_anim)
Li_init = compute_L_inner(hist_init_anim)
Li_opt  = compute_L_inner(hist_opt_anim)

print(f'L (all particles)   — initial: {L_init:.4f}  optimised: {L_opt:.4f}  '
      f'  improvement: {100*(L_init-L_opt)/L_init:.1f}%')
print(f'L (inner sq only)   — initial: {Li_init:.4f}  optimised: {Li_opt:.4f}')

In [ ]:
# Single-run animation (optimised boundary)
ani_single = animate_simulation(
    hist_opt_anim, C_opt,
    box_half=BOX_HALF, inner_half=INNER_HALF,
    title='Optimised boundary — particle simulation',
    interval=180
)
HTML(ani_single.to_jshtml())


In [ ]:
# Side-by-side comparison: initial vs optimised
ani_cmp = animate_comparison(
    hist_init_anim, C_init,
    hist_opt_anim,  C_opt,
    box_half=BOX_HALF, inner_half=INNER_HALF,
    label_a=f'Initial boundary  (L={compute_L(hist_init_anim):.4f})',
    label_b=f'Optimised boundary (L={compute_L(hist_opt_anim):.4f})',
    interval=180
)
HTML(ani_cmp.to_jshtml())


## 1-D Objective Landscape
Sweep $L$ along the gradient direction at $C_{\mathrm{opt}}$ and along the $c_0$ axis.
A **local minimum at $t=0$** confirms the optimised boundary is (at least locally) optimal.


In [ ]:
# Define a quick L-evaluator with fixed particles
def quick_L(C_, seed=0, n_avg=3):
    """Average L over n_avg runs to reduce stochasticity."""
    vals = []
    for s in range(n_avg):
        sim_ = ForwardSimulation(C_, DT, n_coll_pairs=N_COLL, seed=seed+s)
        h_   = sim_.run(x0.copy(), v0.copy(), N_STEPS)
        vals.append(compute_L(h_))
    return float(np.mean(vals))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Sweep along c0 (mean radius)
d_c0 = np.zeros_like(C_opt); d_c0[0] = 1.0
plot_objective_landscape_1d(
    C_opt, d_c0, (-0.3, 0.3), quick_L, n_points=15, ax=ax1
)
ax1.set_title('Sweep along $c_0$ (mean radius)')

# Sweep along the shape gradient at the optimum
sim_tmp  = ForwardSimulation(C_opt, DT, n_coll_pairs=N_COLL, seed=0)
hist_tmp = sim_tmp.run(x0.copy(), v0.copy(), N_STEPS)
b_tmp, a_tmp = hist_tmp.backward_pass(term_beta, term_alpha)
g_at_opt = shape_gradient(hist_tmp, b_tmp, a_tmp)
if np.linalg.norm(g_at_opt) > 1e-10:
    d_grad = g_at_opt / np.linalg.norm(g_at_opt)
    plot_objective_landscape_1d(
        C_opt, d_grad, (-0.3, 0.3), quick_L, n_points=15, ax=ax2
    )
    ax2.set_title('Sweep along gradient direction at $C_{\\mathrm{opt}}$')

plt.tight_layout(); plt.show()


## Final Particle Density Heat Maps
2-D histograms of final positions confirm whether the optimised boundary
steers more particles toward (or within) the inner square.


In [ ]:
fig_density = plot_final_density(
    hist_init_anim, hist_opt_anim,
    BOX_HALF, INNER_HALF,
    label_a=f'Initial  (L={compute_L(hist_init_anim):.4f})',
    label_b=f'Optimised (L={compute_L(hist_opt_anim):.4f})',
)
plt.show()

print('C (initial): ', C_init.round(4))
print('C (optimised):', C_opt.round(4))
print(f'Perimeter: {perimeter(C_init):.4f} → {perimeter(C_opt):.4f}  (target {P_TARGET:.4f})')
print(f'L (all particles): {compute_L(hist_init_anim):.4f} → {compute_L(hist_opt_anim):.4f}')
print(f'r(theta) range: min={min(radius_r(t,C_opt) for t in np.linspace(0,1,200)):.4f}, '
      f'max={max(radius_r(t,C_opt) for t in np.linspace(0,1,200)):.4f}')